# Missão: Decifrando o Absenteísmo em um Serviço de Reabilitação

**TCC:** Predição de No-show em Sistemas de Agendamento: Um Framework Probabilístico com Normalização de Dados Heterogêneos
**Autores:** Cesar Augusto da Silva Pisa e Natan Felipe Kainak Arend · **Orientador:** Prof. Vinicius Tessele

---

## O cenário

Um serviço ambulatorial de reabilitação vinculado ao SUS atende crianças e adultos com deficiência intelectual e
motora (fisioterapia, fonoaudiologia, psicoterapia, terapia ocupacional etc.). Cada falta sem aviso (*no-show*) é
um horário de terapeuta perdido e uma vaga que outro paciente da fila poderia ter usado.

A pergunta que guia esta análise exploratória é:

> **"Quem falta, quando falta e o que os dados disponíveis conseguem (e não conseguem) dizer sobre isso?"**

As respostas alimentam diretamente as decisões do framework de pré-processamento e da Rede Bayesiana do TCC.

---

## Dataset

**Nome:** *Medical appointments no-show* (serviço de reabilitação, região de Itajaí/SC), versão em inglês
**Arquivo:** `data/raw/medical-appointments-no-show-en.csv` (49.593 agendamentos, jun/2016 a fev/2022)

| Coluna | Significado |
|--------|-------------|
| `specialty` | Especialidade do atendimento |
| `appointment_date` / `appointment_time` | Data e horário da consulta |
| `no_show` | **Alvo**: `yes` = faltou, `no` = compareceu |
| `no_show_reason` | Motivo registrado da falta (texto livre) |
| `gender`, `date_of_birth`, `age`, `city` | Perfil do paciente |
| `disability`, `icd` | Tipo de deficiência e CID-10 |
| `entry_service_date` | Data de entrada do paciente no serviço |
| `patient_needs_companion` | Paciente precisa de acompanhante |
| `average_temp_day`, `max_rain_day`, `rain_intensity`, `heat_intensity`... | Clima do dia da consulta |
| `rainy_day_before`, `storm_day_before` | Chuva/tempestade no dia anterior |

---

## Como este notebook está organizado

O roteiro segue a estrutura do desafio de EDA da disciplina: cada atividade tem um **briefing** (o que investigar e
por quê), o código, e um **checkpoint** com os achados registrados em texto.

> **Dica:** sempre perguntem "por que isso acontece?" antes de passar para o próximo passo. Um gráfico sem
> interpretação é apenas uma imagem.

---

## Atividade 1: Preparação do Ambiente

O projeto roda localmente no ambiente virtual `.venv` (kernel **"TCC no-show (.venv)"**), com as dependências de
`requirements.txt`. Não é necessário Colab nem upload manual: o caminho da base é resolvido a partir da raiz do projeto.

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from scipy import stats

from noshow.weather import fix_weather, load_openmeteo, heat_category, rain_category

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CSV = RAIZ / "data" / "raw" / "medical-appointments-no-show-en.csv"

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", "{:.3f}".format)

# Paleta: azul = magnitude / compareceu, laranja = no-show; texto e grade neutros.
BLUE, ORANGE, GRAY, INK = "#2a78d6", "#eb6834", "#8a8984", "#52514e"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
CLASSES = {0: ("compareceu", BLUE), 1: ("no-show", ORANGE)}
DIVERGENTE = LinearSegmentedColormap.from_list("div", [ORANGE, "#f2f1ed", BLUE])
PCT = mtick.PercentFormatter(1, decimals=0)

plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": GRAY, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK,
    "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
})

df = pd.read_csv(CSV)
df.head()

---

## Atividade 2: Conhecendo o Dataset

### Briefing

Antes de qualquer gráfico, o reconhecimento do terreno:

- Quantos agendamentos e quantas informações por agendamento?
- Quais colunas existem e quais são seus tipos? Os tipos fazem sentido (datas como texto?)
- Há informações faltando? Onde? Quanto? A ausência parece aleatória ou sistemática?
- Qual é o perfil geral dos dados numéricos?

> **Atenção aos "vazios disfarçados":** no TMDB eram os zeros em `budget`. Aqui, o equivalente são **textos em
> branco** (`" "`), categorias genéricas (`"sem especialidade"`), códigos indeterminados (`gender = "I"`) e **datas
> digitadas errado** que o pandas não consegue converter. Nenhum deles aparece em `.isna()`.

In [ ]:
n_linhas, n_colunas = df.shape
print(f"Agendamentos (linhas): {n_linhas:,}")
print(f"Atributos por agendamento (colunas): {n_colunas}")
print()
df.info()

In [ ]:
faltantes = pd.DataFrame({
    "nulos": df.isna().sum(),
    "nulos_%": (df.isna().mean() * 100).round(2),
    "valores_únicos": df.nunique(),
}).sort_values("nulos_%", ascending=False)
faltantes

In [ ]:
pct = faltantes["nulos_%"].sort_values()
pct = pct[pct > 0]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(pct.index, pct.values, color=BLUE, height=0.6)
for i, v in enumerate(pct.values):
    ax.text(v + 0.8, i, f"{v:.1f}%", va="center", fontsize=8, color=INK)
ax.set_title("Percentual de valores ausentes por coluna (apenas colunas com ausência)")
ax.set_xlabel("% de registros")
ax.set_xlim(0, 105)
ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

In [ ]:
# Vazios disfarçados: valores que não são NaN, mas não carregam informação.
texto = df.select_dtypes(include=["object", "string"])
so_espaco = texto.apply(lambda s: s.str.strip().eq("").sum())
print("Células só com espaço em branco:", so_espaco[so_espaco > 0].to_dict())
print("specialty = 'sem especialidade':", (df.specialty == "sem especialidade").sum())
print("gender = 'I' (indeterminado):", (df.gender == "I").sum())

# Datas que existem como texto mas não são datas válidas
for col in ["appointment_date", "date_of_birth", "entry_service_date"]:
    s = df[col].dropna().str.strip()
    invalidas = s[pd.to_datetime(s, format="%d/%m/%Y", errors="coerce").isna()]
    print(f"\n{col}: {len(invalidas)} datas inválidas")
    if len(invalidas):
        print(invalidas.value_counts().head(5).to_string())

In [ ]:
# Vazio disfarçado nas colunas de clima: o que acontece quando a temperatura está ausente?
sem_clima = df.average_temp_day.isna()
print(f"Registros sem dados meteorológicos: {sem_clima.sum():,}")
print("  rain_intensity nesses registros:", df.loc[sem_clima, "rain_intensity"].value_counts().to_dict())
print("  heat_intensity nesses registros:", df.loc[sem_clima, "heat_intensity"].value_counts().to_dict())
print(f"\nrainy_day_before == storm_day_before em {(df.rainy_day_before == df.storm_day_before).mean():.0%} das linhas")
print(pd.crosstab(df.rainy_day_before, sem_clima.map({True: "clima ausente", False: "clima presente"})))
print(f"\nheavy_cold: {(df.heat_intensity == 'heavy_cold').sum():,} registros, dos quais "
      f"{(sem_clima & (df.heat_intensity == 'heavy_cold')).sum():,} são clima ausente")

In [ ]:
# Blocos de ausência: as colunas do cadastro do paciente faltam juntas?
cadastro = ["date_of_birth", "entry_service_date", "city", "age", "disability"]
padrao = df[cadastro].isna().value_counts().head(5)
print("Combinações mais comuns de ausência no cadastro (True = ausente):")
print(padrao.to_string())

sem_cadastro = df[cadastro].isna().all(axis=1)
taxa_sem = df.loc[sem_cadastro, "no_show"].eq("yes").mean()
taxa_com = df.loc[~sem_cadastro, "no_show"].eq("yes").mean()
print(f"\nNo-show SEM cadastro: {taxa_sem:.1%}  |  COM cadastro: {taxa_com:.1%}")

In [ ]:
df.describe().T

**Checkpoint: registre aqui o que vocês descobriram:**

1. Quantos agendamentos e atributos o dataset tem? O alvo está balanceado?
2. Quais colunas têm mais dados ausentes? Esse padrão parece aleatório ou sistemático?
3. Que "vazios disfarçados" vocês encontraram?
4. Algo nos valores mínimos ou máximos chamou a atenção? Cite pelo menos um caso.

---

1. São **49.593 agendamentos × 26 atributos**, entre jun/2016 e fev/2022. O alvo é bem desbalanceado: **4.832 no-shows
   (9,7%)**, cerca de 9,3 comparecimentos para cada falta.

2. A ausência é **sistemática**, não aleatória:
   - `no_show_reason` (96,5% vazio) é vazio por construção: só é preenchido quando há falta.
   - `icd` (78,4%) só existe para uma parte dos pacientes.
   - `age`/`date_of_birth` (~20,8%) e o bloco de cadastro (`city`, `entry_service_date`, `disability`, ~10,4%)
     faltam **juntos**. São 5.137 agendamentos sem nenhum dado do paciente, e eles têm **15,1% de no-show contra
     9,1%** dos que têm cadastro. A ausência carrega informação: é um indicativo de *missing not at random*.
   - O clima (2,0% ausente) falta só em nov–dez/2020 e jan–fev/2022.

3. Vazios disfarçados encontrados:
   - textos só com espaço (9 em `disability`, 16 em `date_of_birth`);
   - `"sem especialidade"` (57) e `gender = "I"` (7);
   - **389 datas inválidas**, como `31/07/201`, `17/0/2017`, `31/06/2018` e intervalos do tipo `12/02/2018 - 27/08/2019`;
   - o mais grave: nos 1.016 dias sem dados meteorológicos, a base preencheu `heat_intensity = "heavy_cold"` e
     `rain_intensity = "no_rain"`. Além disso, `rainy_day_before`/`storm_day_before` são idênticas entre si e só valem 0
     nesses dias. Ou seja, **não medem chuva: são um indicador de clima ausente**. Dos 1.486 registros de "frio
     intenso", 1.016 são, na verdade, dado faltante.

4. `age` vai até 110 anos, e a mediana de 11 anos não bate com a mediana da idade calculada na consulta (Atividade 3).
   `rainy_day_before` tem média 0,981, ou seja, é praticamente constante. `max_rain_day` chega a 45 mm, com mediana de
   0,2 mm, o que indica cauda longa.

---

## Atividade 3: Limpeza e Preparação dos Dados

### Briefing

*Garbage in, garbage out.* Neste bloco:

1. Tratar os vazios disfarçados (espaços → `NaN`, datas inválidas → `NaT`).
2. Converter as datas (formato brasileiro `dd/mm/aaaa`) e extrair **ano, mês, dia da semana e hora**.
3. Fazer o *parse* de dois campos "sujos":
   - `icd`: o mesmo diagnóstico aparece com e sem subcategoria (`F84` e `F84.0`), então é preciso agrupar pela
     categoria de 3 caracteres;
   - `no_show_reason`: texto livre, que deve ser agrupado em categorias de motivo.
4. Criar variáveis derivadas: **idade na data da consulta**, **dias em serviço** e **ID aproximado do paciente**.

### Sobre a coluna `age`

Ela parece óbvia, mas **não é a idade na data da consulta**. O teste abaixo mostra que `age` foi calculada na data
de extração da base (~2022). Para uma consulta de 2016, ela superestima a idade em até 6 anos, e as colunas
`under_12_years_old`/`over_60_years_old` herdam o erro. A idade correta precisa ser recalculada a partir de
`date_of_birth`.

### Sobre `no_show_reason`

O motivo só é registrado **depois** que a falta acontece. Ele é ótimo para entender o fenômeno, mas **não pode ser
usado como preditor** (seria vazamento de dados: o modelo "veria o futuro").

In [ ]:
base = df.copy()

# 1. Vazios disfarçados
texto = base.select_dtypes(include=["object", "string"]).columns
base[texto] = base[texto].apply(lambda s: s.str.strip()).replace("", np.nan)
base["gender"] = base["gender"].replace("I", np.nan)

# 2. Datas e variáveis de calendário
for col in ["appointment_date", "date_of_birth", "entry_service_date"]:
    base[col] = pd.to_datetime(base[col], format="%d/%m/%Y", errors="coerce")

base["no_show"] = base["no_show"].eq("yes").astype(int)

# Clima: remove os valores de preenchimento e completa a temperatura dos 32 dias sem medição (ver validação abaixo)
base = fix_weather(base)
base["ano"] = base.appointment_date.dt.year
base["mes"] = base.appointment_date.dt.month
DIAS = ["seg", "ter", "qua", "qui", "sex", "sáb", "dom"]
base["dia_semana"] = pd.Categorical(base.appointment_date.dt.dayofweek.map(dict(enumerate(DIAS))),
                                    categories=DIAS, ordered=True)
base["hora"] = base.appointment_time.str[:2].astype(int)

print(base[["appointment_date", "ano", "mes", "dia_semana", "appointment_time", "hora"]].head())

In [ ]:
# Teste: `age` é a idade na consulta ou na extração da base?
idade_consulta = (base.appointment_date - base.date_of_birth).dt.days / 365.25
idade_2022 = (pd.Timestamp("2022-03-01") - base.date_of_birth).dt.days / 365.25
comparacao = pd.DataFrame({
    "age − idade na consulta": (base.age - idade_consulta).describe(),
    "age − idade em mar/2022": (base.age - idade_2022).describe(),
}).loc[["mean", "min", "50%", "max"]]
print(comparacao.round(2))

erro_por_ano = (base.age - idade_consulta).groupby(base.ano).mean()
print("\nErro médio de `age` por ano da consulta (anos):")
print(erro_por_ano.round(1).to_string())

### Sobre o clima: validação externa e preenchimento

Os 1.016 agendamentos sem medição (32 dias, quase todos em jan–fev/2022) estavam rotulados como "frio intenso" e
"sem chuva". Para decidir como tratá-los, a base foi comparada ao histórico do **Open-Meteo** (reanálise ERA5) para
Itajaí, arquivo `data/external/openmeteo_itajai_diario.csv`:

1. **As categorias são regras fixas:** `heat_intensity` são cortes em `max_temp_day` (15/22/27/32 °C) e
   `rain_intensity` são cortes em `max_rain_day` (1/2,5/10 mm/h).
2. **Temperatura confere** com o Open-Meteo, então é seguro usá-la para preencher os dias ausentes.
3. **Chuva não confere:** o ERA5 suaviza pancadas localizadas, então a chuva desses dias **permanece ausente**.

A função `fix_weather` (em `src/noshow/weather.py`) aplica a correção e registra a origem em `clima_fonte`.

In [ ]:
om = load_openmeteo()
dia = df.assign(dt=pd.to_datetime(df.appointment_date, format="%d/%m/%Y")).groupby("dt")[
    ["average_temp_day", "max_temp_day", "average_rain_day", "max_rain_day", "heat_intensity", "rain_intensity"]].first()
medidos = dia[dia.average_temp_day.notna()].join(om, rsuffix="_om")

print("1. As regras reproduzem as categorias da base?")
print(f"   heat_intensity: {(heat_category(medidos.max_temp_day) == medidos.heat_intensity).mean():.0%} | "
      f"rain_intensity: {(rain_category(medidos.max_rain_day) == medidos.rain_intensity).mean():.0%}")

validacao = pd.DataFrame({
    nome: {"correlação": medidos[c].corr(medidos[f"{c}_om"]),
           "erro abs. mediano": (medidos[c] - medidos[f"{c}_om"]).abs().median()}
    for nome, c in [("temp. média (°C)", "average_temp_day"), ("temp. máxima (°C)", "max_temp_day"),
                    ("chuva média (mm/h)", "average_rain_day"), ("chuva máx. horária (mm/h)", "max_rain_day")]
}).T
print(f"\n2–3. Base × Open-Meteo nos {len(medidos)} dias medidos:")
print(validacao.round(3).to_string())

cat_om = heat_category(medidos.max_temp_day_om)
ordem = {c: i for i, c in enumerate(["heavy_cold", "cold", "mild", "warm", "heavy_warm"])}
print(f"\nheat_intensity recalculada com Open-Meteo: {(cat_om == medidos.heat_intensity).mean():.0%} idêntica, "
      f"{((cat_om.map(ordem) - medidos.heat_intensity.map(ordem)).abs() <= 1).mean():.0%} a até uma categoria")
print(f"rain_intensity recalculada com Open-Meteo: {(rain_category(medidos.max_rain_day_om) == medidos.rain_intensity).mean():.0%} idêntica")

ausentes = dia[dia.average_temp_day.isna()].join(om, rsuffix="_om")
print(f"\nNos {len(ausentes)} dias sem medição, o Open-Meteo registra máxima entre "
      f"{ausentes.max_temp_day_om.min():.1f} e {ausentes.max_temp_day_om.max():.1f} °C "
      f"→ nenhum dia de frio intenso (≤ 15 °C)")
print("\nResultado do preenchimento:", base.clima_fonte.value_counts(dropna=False).to_dict())
print("heat_intensity nos dias preenchidos:", base.loc[base.clima_fonte == "open-meteo", "heat_intensity"].value_counts().to_dict())

In [ ]:
# 4. Variáveis derivadas
base["idade_consulta"] = idade_consulta.round(1)
base["dias_em_servico"] = (base.appointment_date - base.entry_service_date).dt.days

# ID aproximado: a base não tem identificador de paciente; atributos estáveis do cadastro fazem esse papel.
CHAVE = ["date_of_birth", "gender", "city", "entry_service_date", "disability"]
completa = base[CHAVE].notna().all(axis=1)
base["patient_id"] = pd.NA
base.loc[completa, "patient_id"] = base.loc[completa].groupby(CHAVE).ngroup()

print(f"Idade na consulta: mediana {base.idade_consulta.median():.1f} anos")
print(f"Dias em serviço: mediana {base.dias_em_servico.median():.0f} | negativos: {(base.dias_em_servico < 0).sum()}")
print(f"Registros com ID aproximado: {completa.mean():.1%} → {base.patient_id.nunique():,} pacientes")

In [ ]:
# 3a. Parse do CID-10: categoria de 3 caracteres + capítulo
base["cid3"] = base.icd.str.upper().str.replace(r"[^A-Z0-9]", "", regex=True).str[:3]
CAPITULOS = {"F": "F — transtornos mentais/comportamentais", "G": "G — doenças do sistema nervoso",
             "Q": "Q — malformações congênitas", "R": "R — sintomas e sinais anormais",
             "I": "I — doenças do aparelho circulatório"}
base["cid_capitulo"] = base.cid3.str[0].map(CAPITULOS).fillna(base.cid3.str[0] + " — outros")

print(f"CIDs distintos: {base.icd.nunique()} originais → {base.cid3.nunique()} categorias de 3 caracteres")
print(base.cid3.value_counts().head(8).to_string())

In [ ]:
# 3b. Parse de no_show_reason: texto livre → categoria de motivo (por palavras-chave)
REGRAS = {
    "saúde": r"doen|gripe|febre|covid|atestad|hospital|resfriad|viros|vacina|consulta m|médic|medic|exame|"
             r"cirurg|dor|tosse|vomit|diarr|internad|quarentena|machuc|crise|surto",
    "transporte": r"transporte|carro|ônibus|onibus|van|carona|chuva",
    "agenda / cancelamento": r"desmarc|remarc|cancel|avisou|desist|transferid|alta|feriado|agenda",
    "compromisso / viagem": r"compromisso|viaj|viagem|escola|trabalh|aula",
    "registro sem sentido": r"^ok|^yes|atendeu|^sim$|^não$",
}

def categoriza_motivo(texto):
    if pd.isna(texto):
        return np.nan
    t = texto.lower()
    for categoria, padrao in REGRAS.items():
        if re.search(padrao, t):
            return categoria
    return "outros"

base["motivo_cat"] = base.no_show_reason.map(categoriza_motivo)
motivos = base.loc[base.no_show == 1, "motivo_cat"].value_counts(dropna=False).rename(index={np.nan: "(sem motivo)"})
print("Motivos registrados entre os no-shows:")
print(pd.DataFrame({"n": motivos, "%": (motivos / motivos.sum() * 100).round(1)}))

In [ ]:
# Segmentações: faixas etárias (marcos de desenvolvimento/atendimento) e quintis de tempo de serviço
base["faixa_etaria"] = pd.cut(base.idade_consulta, bins=[0, 6, 12, 18, 60, np.inf], right=False,
                              labels=["0–5 (primeira infância)", "6–11 (criança)", "12–17 (adolescente)",
                                      "18–59 (adulto)", "60+ (idoso)"])
validos = base.dias_em_servico >= 0
base.loc[validos, "quintil_servico"] = pd.qcut(base.loc[validos, "dias_em_servico"], 5,
                                               labels=["Q1 (recém-chegado)", "Q2", "Q3", "Q4", "Q5 (veterano)"])

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
for ax, col in zip(axes, ["faixa_etaria", "quintil_servico"]):
    cont = base[col].value_counts().sort_index()
    ax.bar(cont.index.astype(str), cont.values, color=BLUE, width=0.6)
    ax.set_title(f"Agendamentos por {col.replace('_', ' ')}")
    ax.tick_params(axis="x", labelsize=8, rotation=15)
plt.tight_layout(); plt.show()

print(base.groupby("quintil_servico", observed=True).dias_em_servico.agg(["min", "max"]))

In [ ]:
# Base analítica: sem colunas de vazamento e com a idade corrigida no lugar de `age`
VAZAMENTO = ["no_show_reason", "motivo_cat"]
REDUNDANTES = ["age", "under_12_years_old", "over_60_years_old", "appointment_month", "appointment_year"]
analise = base.drop(columns=VAZAMENTO + REDUNDANTES)
print("Shape base completa:", base.shape, "| shape base analítica:", analise.shape)
analise.head()

**Checkpoint: registre aqui o que vocês descobriram:**

1. Que tratamentos foram aplicados aos vazios disfarçados? Existe alguma desvantagem nessa abordagem?
2. A coluna `age` é confiável? O que o teste mostrou e qual foi a decisão?
3. Quantas categorias de CID restaram após o parse? Quais dominam e o que isso diz sobre o público atendido?
4. Quais são os principais motivos de falta registrados? Por que essa coluna não pode entrar no modelo?
5. Que critérios foram usados para as faixas etárias e os quintis? Que outras formas de segmentar existiriam?

---

1. Os textos foram limpos (`strip`), espaços viraram `NaN`, `gender = "I"` virou `NaN` e as datas inválidas viraram
   `NaT`. No clima, os valores de preenchimento foram removidos e as duas colunas de "dia anterior" foram descartadas
   (só variavam nos dias sem medição). A **temperatura dos 32 dias sem medição foi preenchida com o Open-Meteo**, que
   confere com a base nos dias medidos (r = 0,98). A **chuva desses dias ficou ausente**, porque o Open-Meteo não é
   confiável para chuva. A origem de cada valor fica registrada em `clima_fonte`.
   **Desvantagem:** as 389 datas digitadas errado são perdidas. Algumas poderiam ser recuperadas manualmente, mas
   `31/07/201` é ambígua. A ausência também precisa virar uma categoria explícita na modelagem, senão os 10% de
   registros sem cadastro, que faltam mais, somem da análise.

2. **Não é confiável.** `age` difere da idade na data da consulta em 4,05 anos em média, mas bate com a idade em
   mar/2022 (diferença média de 0,3 ano). Ela foi calculada **na extração da base**. O erro é de 5,7 anos para consultas
   de 2016 e de 0,4 ano em 2022. Como consequência, `under_12_years_old` e `over_60_years_old` também estão erradas para
   os anos antigos. **Decisão:** usar `idade_consulta`, recalculada a partir de `date_of_birth`, e descartar as três
   colunas originais.

3. Os 63 códigos originais viraram **46 categorias de 3 caracteres**. Dominam **F84** (transtornos globais do
   desenvolvimento/TEA, 5.388 registros) e **R68** (sintomas gerais, 2.491), seguidos de F80 (fala e linguagem) e
   I64/I69 (AVC e sequelas). O público é majoritariamente infantil e neurodivergente, com um grupo adulto de
   reabilitação pós-AVC. Atenção: o CID existe para apenas 22% dos registros.

4. Em **65,3% dos no-shows nenhum motivo foi registrado**. Entre os registrados, predominam **saúde** (14,7% dos
   no-shows: doença, gripe, febre, COVID), **agenda/cancelamento** (5,8%) e **transporte** (5,6%). Há ruído: 60
   registros com motivo pertencem a pacientes que *compareceram* ("yes atendeu", "ok mãe"). Além disso, motivos como
   "desmarcado" e "avisou" mostram que o rótulo `no_show` **mistura faltas sem aviso com cancelamentos**, o que diverge
   da definição da Seção 2.1.1. A coluna não pode entrar no modelo porque **só existe depois da falta**: seria
   vazamento, e o modelo "veria o futuro".

5. As **faixas etárias** seguem marcos de desenvolvimento e de atendimento (primeira infância 0–5, criança 6–11,
   adolescente 12–17, adulto, idoso 60+). O **tempo de serviço** foi dividido em quintis (`qcut`, mesma frequência por
   faixa): Q1 vai de 0 a 141 dias e Q5 de 429 a 2.776 dias. Alternativas seriam tercis, faixas fixas em meses ou uma
   **discretização supervisionada** (por exemplo, MDLP ou cortes de árvore de decisão), que escolhe os cortes que
   melhor separam o no-show. Esta última é relevante para a Rede Bayesiana do TCC.

---

## Atividade 4: Investigação Univariada: Entendendo Cada Variável

### Briefing

Cada variável, separadamente. **Por que isso importa?** Se `dias_em_servico` tem cauda longa (poucos pacientes com
anos de vínculo), a média engana e a mediana é mais honesta. E, para o TCC, a forma da distribuição decide **qual
escalonador** faz sentido (MinMax é sensível a outliers; Robust usa mediana e IQR) e **como discretizar** para a
Rede Bayesiana.

**Conceitos:** histograma + KDE, boxplot, assimetria (*skewness*), curtose e escala logarítmica (`np.log1p`).

**O que investigar:**

- `idade_consulta`, `dias_em_servico`, temperatura e chuva
- O alvo `no_show`: qual é o desbalanceamento?
- As categóricas: quais especialidades, deficiências, cidades e CIDs dominam?

> **Perguntas-guia:** Qual é o paciente típico? A chuva é um fenômeno frequente ou raro? Há valores claramente
> inconsistentes? Qual é a taxa de no-show?

In [ ]:
taxa = analise.no_show.mean()
contagem = analise.no_show.value_counts().sort_index()

fig, ax = plt.subplots(figsize=(6, 2.2))
ax.barh(["compareceu", "no-show"], contagem.values, color=[BLUE, ORANGE], height=0.55)
for i, v in enumerate(contagem.values):
    ax.text(v, i, f"  {v:,} ({v / contagem.sum():.1%})", va="center", fontsize=9, color=INK)
ax.set_xlim(0, contagem.max() * 1.3); ax.grid(axis="y", visible=False)
ax.set_title("Distribuição do alvo")
plt.tight_layout(); plt.show()
print(f"Desbalanceamento: {contagem[0] / contagem[1]:.1f} comparecimentos para cada no-show")

In [ ]:
NUMERICAS = ["idade_consulta", "dias_em_servico", "average_temp_day", "max_temp_day", "average_rain_day", "max_rain_day"]
ASSIMETRICAS = ["idade_consulta", "dias_em_servico", "average_rain_day", "max_rain_day"]

fig, axes = plt.subplots(len(ASSIMETRICAS), 2, figsize=(11, 10))
for linha, col in zip(axes, ASSIMETRICAS):
    s = analise[col].dropna()
    s_log = np.log1p(s.clip(lower=0))
    sns.histplot(s, bins=50, kde=True, color=BLUE, ax=linha[0], edgecolor="white", linewidth=0.5)
    sns.histplot(s_log, bins=50, kde=True, color=BLUE, ax=linha[1], edgecolor="white", linewidth=0.5)
    linha[0].set_title(f"{col} — escala original")
    linha[1].set_title(f"{col} — log1p")
    for ax in linha:
        ax.set_xlabel(""); ax.set_ylabel("")
plt.tight_layout(); plt.show()

In [ ]:
forma = pd.DataFrame({
    col: {"média": s.mean(), "mediana": s.median(), "desvio": s.std(), "assimetria": s.skew(),
          "curtose": s.kurtosis(), "% outliers (IQR)": 100 * ((s < s.quantile(.25) - 1.5 * (s.quantile(.75) - s.quantile(.25))) |
                                                               (s > s.quantile(.75) + 1.5 * (s.quantile(.75) - s.quantile(.25)))).mean(),
          "% zeros": 100 * (s == 0).mean()}
    for col in NUMERICAS for s in [analise[col].dropna()]
}).T
forma

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 2.8))
for ax, col in zip(axes, ["idade_consulta", "dias_em_servico"]):
    sns.boxplot(x=analise[col], ax=ax, color=BLUE, fliersize=2, linewidth=1, width=0.4,
                flierprops={"markerfacecolor": GRAY, "markeredgecolor": GRAY})
    ax.set_title(f"Outliers — {col}"); ax.set_xlabel("")
plt.tight_layout(); plt.show()

print("Consultas ANTES da entrada no serviço (dias_em_servico < 0):", (analise.dias_em_servico < 0).sum())
print("Pacientes com idade na consulta ≥ 100:", (analise.idade_consulta >= 100).sum())
print("Consultas em fim de semana:", analise.dia_semana.isin(["sáb", "dom"]).sum())
print("Consultas às 12h (intervalo de almoço):", (analise.hora == 12).sum())

In [ ]:
CATEGORICAS = ["specialty", "disability", "city", "cid_capitulo", "faixa_etaria", "patient_needs_companion"]
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, col in zip(axes.flat, CATEGORICAS):
    cont = analise[col].astype("string").fillna("(ausente)").value_counts().head(8).sort_values()
    ax.barh(cont.index, cont.values, color=BLUE, height=0.6)
    for i, v in enumerate(cont.values):
        ax.text(v, i, f" {v / len(analise):.0%}", va="center", fontsize=7, color=INK)
    ax.set_title(col); ax.grid(axis="y", visible=False)
    ax.tick_params(axis="y", labelsize=8); ax.set_xlim(0, cont.max() * 1.25)
fig.suptitle("Composição das variáveis categóricas (top 8 categorias, % do total)", x=0.01, ha="left", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
consultas_paciente = analise.dropna(subset=["patient_id"]).groupby("patient_id").size()
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(consultas_paciente, bins=40, color=BLUE, edgecolor="white", linewidth=1)
ax.axvline(consultas_paciente.median(), color=GRAY, ls="--", lw=1)
ax.text(consultas_paciente.median(), ax.get_ylim()[1] * 0.9, f"  mediana = {consultas_paciente.median():.0f}", fontsize=8, color=INK)
ax.set_title("Agendamentos por paciente (ID aproximado)")
ax.set_xlabel("agendamentos"); ax.set_ylabel("pacientes")
plt.tight_layout(); plt.show()

**Checkpoint: registre aqui o que vocês descobriram:**

1. Qual é o perfil típico do paciente (idade, deficiência, cidade, especialidade)? Use mediana e moda.
2. Qual é a taxa de no-show e o grau de desbalanceamento? Que consequência isso tem para um classificador?
3. Quais variáveis contínuas são fortemente assimétricas? A escala log ajudou? O que isso implica para a escolha do escalonador?
4. Que valores inconsistentes vocês encontraram?
5. Como os agendamentos se distribuem por paciente? Por que isso importa para a validação do modelo?

---

1. O paciente típico é uma **criança de ~7 anos** (mediana da idade na consulta: 6,9; 78% dos pacientes com idade
   conhecida têm menos de 12 anos), com **deficiência intelectual** (62% dos agendamentos), de **Itajaí** (41%),
   atendida em **fonoaudiologia** ou **psicoterapia** (~24% cada) e que **precisa de acompanhante** (54%).

2. A taxa é de **9,7%** (9,3:1). Um classificador ingênuo que diz "todos comparecem" acerta 90,3% e não serve para
   nada. Por isso a acurácia não pode ser a métrica do TCC: o foco é PR-AUC, *recall* e o balanceamento (SMOTENC)
   aplicado só no treino.

3. São fortemente assimétricas:
   - `idade_consulta` (assimetria 2,1);
   - `dias_em_servico` (1,9, com curtose 12,7);
   - a chuva (assimetria 4,3–4,5 e **40% de zeros**).

   O `log1p` deixa idade e tempo de serviço mais legíveis, mas não resolve a chuva, que é inflada de zeros e funciona
   melhor como categoria (`rain_intensity`). Para o escalonamento, as caudas e os 6–17% de outliers pelo critério do
   IQR favorecem o **RobustScaler**, porque o MinMax seria comprimido pelos extremos. Lembrete do notebook 01: isso
   afeta a distância usada pelo SMOTENC, não as faixas da Rede Bayesiana.

4. Inconsistências encontradas:
   - **850 consultas antes da entrada no serviço** (`dias_em_servico` < 0), provavelmente reingressos com a data de
     entrada sobrescrita;
   - **84 registros com idade ≥ 100 anos**, implausíveis e provavelmente data de nascimento digitada errado;
   - 1.301 consultas em fim de semana;
   - 105 consultas ao meio-dia.

5. Cada paciente tem **mediana de 15 agendamentos**, e há pacientes com até 178. Se a validação cruzada embaralhar os
   registros, o mesmo paciente aparece no treino e no teste, e atributos do cadastro funcionam como "impressão
   digital". Isso infla as métricas. A validação precisa **agrupar por paciente** (`StratifiedGroupKFold`).

---

## Atividade 5: Investigação Bivariada e Multivariada: Cruzando as Pistas

### Briefing

Agora cada variável é cruzada com o alvo e com as outras. Como o alvo é **binário**, a ferramenta principal é a
**taxa de no-show por grupo**, comparada com a média geral.

**Correlação (Pearson × Spearman):** Pearson mede associação linear e é sensível a outliers; Spearman mede
associação monotônica e é mais robusta para as distribuições assimétricas daqui. Correlação com um alvo 0/1
equivale à correlação ponto-bisserial.

**Importante:** correlação **não é causalidade**. Exemplo: se pacientes que precisam de acompanhante faltam mais,
a causa pode ser a idade (crianças pequenas), e não o acompanhante em si. Separar esse tipo de efeito é justamente
o papel do Manto de Markov no TCC.

**Perguntas centrais:**

- Quais especialidades, deficiências e cidades faltam mais?
- Horário e dia da semana importam?
- Pacientes novos no serviço faltam mais ou menos que os veteranos?
- O clima (chuva, frio) muda o comparecimento?
- As faltas estão concentradas em poucos pacientes?

In [ ]:
corr_cols = ["no_show", "idade_consulta", "dias_em_servico", "hora", "average_temp_day", "max_temp_day",
             "average_rain_day", "max_rain_day", "patient_needs_companion"]
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, metodo in zip(axes, ["pearson", "spearman"]):
    c = analise[corr_cols].corr(method=metodo)
    sns.heatmap(c, annot=True, fmt=".2f", cmap=DIVERGENTE, vmin=-1, vmax=1, ax=ax, cbar=False,
                annot_kws={"size": 7}, linewidths=1, linecolor="white", square=True)
    ax.set_title(metodo.capitalize()); ax.tick_params(labelsize=8)
plt.tight_layout(); plt.show()

dif = (analise[corr_cols].corr("spearman") - analise[corr_cols].corr("pearson")).abs()
par = dif.where(np.triu(np.ones(dif.shape, dtype=bool), 1)).stack().sort_values(ascending=False).head(3)
print("Pares com maior diferença Spearman − Pearson:")
print(par.round(3).to_string())

In [ ]:
def taxa_por(col, data=analise, min_n=30):
    g = data.assign(_c=data[col].astype("string").fillna("(ausente)")).groupby("_c").no_show.agg(taxa="mean", n="size")
    return g[g.n >= min_n].sort_values("taxa")

def plota_taxas(colunas, titulo, ncols=3):
    nlin = int(np.ceil(len(colunas) / ncols))
    fig, axes = plt.subplots(nlin, ncols, figsize=(4.4 * ncols, 3.3 * nlin))
    for ax, col in zip(np.ravel(axes), colunas):
        g = taxa_por(col)
        ax.barh(g.index, g.taxa, color=BLUE, height=0.6)
        ax.axvline(taxa, color=GRAY, ls="--", lw=1)
        for i, (t, n) in enumerate(zip(g.taxa, g.n)):
            ax.text(t, i, f" {t:.1%} (n={n:,})", va="center", fontsize=7, color=INK)
        ax.set_title(col); ax.xaxis.set_major_formatter(PCT); ax.set_xlim(0, g.taxa.max() * 1.7)
        ax.grid(axis="y", visible=False); ax.tick_params(axis="y", labelsize=8)
    for ax in np.ravel(axes)[len(colunas):]:
        ax.set_visible(False)
    fig.suptitle(f"{titulo} (linha tracejada = média geral {taxa:.1%}; grupos com n ≥ 30)",
                 x=0.01, ha="left", fontweight="bold")
    plt.tight_layout(); plt.show()

plota_taxas(["specialty", "disability", "city", "cid_capitulo", "faixa_etaria", "gender"], "Taxa de no-show — perfil")

In [ ]:
plota_taxas(["dia_semana", "hora", "appointment_shift", "quintil_servico", "patient_needs_companion", "mes"],
            "Taxa de no-show — agenda e vínculo")

In [ ]:
plota_taxas(["rain_intensity", "heat_intensity", "clima_fonte"], "Taxa de no-show — clima", ncols=3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, col in zip(axes, ["dias_em_servico", "idade_consulta", "average_temp_day"]):
    dados = analise[[col, "no_show"]].dropna()
    sns.violinplot(data=dados, x="no_show", y=col, hue="no_show", palette=[BLUE, ORANGE], legend=False,
                   inner="quartile", ax=ax, cut=0, linewidth=1)
    ax.set_xticks([0, 1], ["compareceu", "no-show"]); ax.set_xlabel(""); ax.set_title(col)
plt.tight_layout(); plt.show()

In [ ]:
amostra = analise[["idade_consulta", "dias_em_servico", "average_temp_day", "max_rain_day", "no_show"]].dropna()
amostra = amostra.groupby("no_show").sample(300, random_state=42)
g = sns.pairplot(amostra, hue="no_show", palette={0: BLUE, 1: ORANGE}, corner=True,
                 plot_kws={"s": 10, "alpha": 0.5, "linewidth": 0}, diag_kws={"common_norm": False}, height=2)
g._legend.set_title(""); [t.set_text(CLASSES[int(t.get_text())][0]) for t in g._legend.texts]
g.figure.suptitle("Pair plot — amostra estratificada (300 por classe)", x=0.01, ha="left", y=1.02, fontweight="bold")
plt.show()

In [ ]:
perfil = analise.groupby("specialty").agg(
    agendamentos=("no_show", "size"),
    taxa_no_show=("no_show", "mean"),
    idade_mediana=("idade_consulta", "median"),
    dias_servico_mediano=("dias_em_servico", "median"),
    pct_acompanhante=("patient_needs_companion", "mean"),
    pct_intelectual=("disability", lambda s: (s == "intellectual").mean()),
).sort_values("agendamentos", ascending=False)
perfil.style.format({"agendamentos": "{:,}", "taxa_no_show": "{:.1%}", "idade_mediana": "{:.1f}",
                     "dias_servico_mediano": "{:.0f}", "pct_acompanhante": "{:.0%}", "pct_intelectual": "{:.0%}"}) \
    .bar(subset="taxa_no_show", color="#f6c9b5")

In [ ]:
# Concentração das faltas por paciente (curva de Lorenz) e casos anômalos
por_paciente = analise.dropna(subset=["patient_id"]).groupby("patient_id").no_show.agg(faltas="sum", agendamentos="size")
por_paciente["taxa"] = por_paciente.faltas / por_paciente.agendamentos
ordenado = por_paciente.faltas.sort_values(ascending=False).to_numpy()
acum = np.cumsum(ordenado) / ordenado.sum()
x = np.arange(1, len(ordenado) + 1) / len(ordenado)

fig, ax = plt.subplots(figsize=(5.5, 4))
ax.plot(x, acum, color=ORANGE, lw=2)
ax.plot([0, 1], [0, 1], color=GRAY, lw=1, ls="--")
for p in [0.05, 0.10, 0.20]:
    y = acum[int(p * len(acum)) - 1]
    ax.plot(p, y, "o", color=ORANGE, ms=6, mec="white", mew=2)
    ax.annotate(f"{p:.0%} dos pacientes → {y:.0%} das faltas", (p, y), xytext=(8, -4),
                textcoords="offset points", fontsize=8, color=INK)
ax.xaxis.set_major_formatter(PCT); ax.yaxis.set_major_formatter(PCT)
ax.set_xlabel("pacientes (ordenados por nº de faltas)"); ax.set_ylabel("faltas acumuladas")
ax.set_title("Concentração das faltas por paciente")
plt.tight_layout(); plt.show()

print(f"Pacientes que nunca faltaram: {(por_paciente.faltas == 0).mean():.1%}")
print("\nTop 5 pacientes com maior taxa de falta (mín. 20 agendamentos):")
print(por_paciente.query("agendamentos >= 20").nlargest(5, "taxa").to_string())
print("\nTop 5 horários com maior taxa de falta (mín. 200 agendamentos):")
print(analise.groupby("appointment_time").no_show.agg(taxa="mean", n="size").query("n >= 200").nlargest(5, "taxa").to_string())

In [ ]:
por_quintil = analise.groupby("quintil_servico", observed=True).no_show.agg(taxa="mean", n="size")
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.bar(por_quintil.index.astype(str), por_quintil.taxa, color=BLUE, width=0.6)
ax.axhline(taxa, color=GRAY, ls="--", lw=1)
for i, t in enumerate(por_quintil.taxa):
    ax.text(i, t, f"{t:.1%}", ha="center", va="bottom", fontsize=8, color=INK)
ax.yaxis.set_major_formatter(PCT)
ax.set_title("Taxa de no-show por quintil de tempo de serviço")
plt.tight_layout(); plt.show()

**Checkpoint: registre aqui o que vocês descobriram:**

1. Pearson e Spearman deram resultados muito diferentes para algum par? O que isso significa?
2. Quais grupos (perfil, agenda, clima) têm a maior e a menor taxa de no-show? A diferença é grande na prática?
3. Pacientes novos ou veteranos faltam mais? Que hipótese explica o padrão?
4. As faltas estão concentradas em poucos pacientes? Que variável isso sugere criar para o modelo?
5. Qual foi o achado mais surpreendente deste bloco? Formulem uma hipótese para explicá-lo.

---

1. Sim. **`idade_consulta` × `patient_needs_companion`** dá −0,13 em Pearson e −0,55 em Spearman. A relação é
   monotônica e forte (quanto mais nova a criança, mais ela precisa de acompanhante), mas não é linear, porque a cauda
   de adultos e idosos achata o Pearson. Na chuva média × máxima, o Pearson (0,84) é derrubado pelos outliers, e o
   Spearman (0,99) mostra que as duas medem praticamente a mesma coisa (redundância).

2. Maior e menor taxa em cada grupo:
   - **Perfil:** os extremos são **cadastro ausente** (15,1%), cidades como Luiz Alves (13,6%) e Ilhota (13,2%),
     contra **Penha** (6,2%), e o CID do capítulo G (sistema nervoso, 15,2%).
   - **Agenda:** **17h** (12,3%, e o horário 17:40 chega a **20,7%**, o dobro da média) contra **18h** (6,8%); sexta
     (11,0%) contra terça (8,8%); **junho** (13,2%) contra **julho** (7,1%).
   - **Clima:** dias amenos (8,5%) contra os demais (10–11,5%).

   A maioria das diferenças é de 2 a 4 pontos percentuais: pequenas individualmente, mas relevantes quando um mesmo
   agendamento acumula vários fatores de risco.

3. Os quatro primeiros quintis ficam entre 8,1% e 8,7%, mas **Q5 (mais de 429 dias de vínculo) sobe para 11,9%**, e o
   tempo de serviço ausente chega a 14,3%. Uma hipótese é a **fadiga do tratamento longo** (perda de motivação ou
   sensação de melhora). Há uma ressalva importante: o tempo de serviço mediano também cresce ano a ano (Atividade 6),
   junto com a taxa de no-show, então parte desse efeito pode ser a deriva temporal. Essa é exatamente a pergunta que o
   Manto de Markov responde.

4. Muito concentradas: **10% dos pacientes respondem por 42% das faltas** e 20% por 65%, enquanto **37,5% nunca
   faltaram**. Isso pede variáveis de **histórico de comparecimento** (nº e taxa de faltas anteriores do paciente),
   calculadas **somente com agendamentos anteriores** à data de cada consulta para não vazar o futuro.

5. O achado mais surpreendente é o do **acompanhante**. No geral, quem precisa de acompanhante falta menos (9,0% contra
   10,7%). Mas a variável é quase determinada pela idade (todas as crianças com menos de 6 anos precisam e nenhum
   adolescente de 12–17 precisa), e **entre crianças de 6–11 anos o efeito se inverte**: com acompanhante, 10,5%; sem,
   7,9%. **Hipótese:** depender de um adulto disponível (horário de trabalho, transporte) aumenta o risco de falta na
   mesma faixa etária. O efeito marginal era só um reflexo da idade. É um caso clássico de confusão, que mostra por
   que a modelagem condicional (Rede Bayesiana) é necessária.

---

## Atividade 6: A Dimensão do Tempo: Como o Serviço Mudou?

### Briefing

O comportamento dos pacientes não é estático: a base cobre quase seis anos, incluindo a pandemia de COVID-19.
Uma análise que ignora o tempo pode esconder tendências e, pior, pode gerar um modelo que só funciona no passado.

**O que investigar:**

- Como a taxa de no-show, o volume de agendamentos, a idade mediana e o tempo de serviço evoluíram por ano?
- Há meses sem registro? Onde?
- A composição das especialidades mudou ao longo dos anos?

> **Cuidado:** anos com pouquíssimos registros (2020 por causa da pandemia e 2022, que vai só até fevereiro) geram
> taxas instáveis. Mencionem isso na interpretação.

In [ ]:
anual = analise.groupby("ano").agg(agendamentos=("no_show", "size"), taxa_no_show=("no_show", "mean"),
                                   idade_mediana=("idade_consulta", "median"),
                                   dias_servico_mediano=("dias_em_servico", "median"),
                                   pacientes=("patient_id", "nunique"))
metricas = [("agendamentos", "Agendamentos", "{:,.0f}"), ("taxa_no_show", "Taxa de no-show", "{:.1%}"),
            ("idade_mediana", "Idade mediana na consulta (anos)", "{:.1f}"),
            ("dias_servico_mediano", "Dias em serviço (mediana)", "{:.0f}")]

fig, axes = plt.subplots(4, 1, figsize=(9, 9), sharex=True)
for ax, (col, titulo, fmt) in zip(axes, metricas):
    ax.plot(anual.index, anual[col], color=BLUE, lw=2, marker="o", ms=7, mec="white", mew=2)
    for x_, y_ in anual[col].items():
        ax.annotate(fmt.format(y_), (x_, y_), xytext=(0, 7), textcoords="offset points", ha="center", fontsize=8, color=INK)
    ax.set_title(titulo); ax.margins(y=0.3)
    if col == "taxa_no_show":
        ax.yaxis.set_major_formatter(PCT)
for ax in axes:
    ax.axvspan(2019.75, 2021.5, color=GRAY, alpha=0.1)
axes[0].text(2019.8, axes[0].get_ylim()[1] * 0.85, "pandemia", fontsize=8, color=INK)
plt.tight_layout(); plt.show()
anual

In [ ]:
mensal = analise.set_index("appointment_date").no_show.resample("MS").agg(["mean", "size"])
periodo = pd.period_range(analise.appointment_date.min(), analise.appointment_date.max(), freq="M")
vazios = sorted(set(periodo.astype(str)) - set(mensal[mensal["size"] > 0].index.to_period("M").astype(str)))
mensal = mensal[mensal["size"] > 0]

fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
axes[0].plot(mensal.index, mensal["mean"], color=BLUE, lw=2)
axes[0].yaxis.set_major_formatter(PCT); axes[0].set_title("Taxa mensal de no-show")
axes[1].bar(mensal.index, mensal["size"], width=20, color=BLUE)
axes[1].set_title("Agendamentos por mês")
plt.tight_layout(); plt.show()
print("Meses sem nenhum registro:", vazios)
print("Meses com menos de 100 registros:", mensal.index[mensal["size"] < 100].strftime("%Y-%m").tolist())

In [ ]:
top_esp = analise.specialty.value_counts().head(5).index
esp = analise.specialty.where(analise.specialty.isin(top_esp), "outras / ausente").fillna("outras / ausente")
comp = pd.crosstab(analise.ano, esp, normalize="index")[list(top_esp) + ["outras / ausente"]]

fig, ax = plt.subplots(figsize=(10, 4))
esquerda = np.zeros(len(comp))
cores = SERIES[:len(top_esp)] + [GRAY]
for col, cor in zip(comp.columns, cores):
    ax.barh(comp.index.astype(str), comp[col], left=esquerda, color=cor, label=col, height=0.65,
            edgecolor="white", linewidth=2)
    esquerda += comp[col].to_numpy()
ax.xaxis.set_major_formatter(PCT); ax.invert_yaxis(); ax.grid(axis="y", visible=False)
ax.legend(ncol=6, fontsize=8, frameon=False, loc="upper left", bbox_to_anchor=(0, -0.1))
ax.set_title("Composição das especialidades por ano")
plt.tight_layout(); plt.show()
comp.style.format("{:.0%}")

**Checkpoint: registre aqui o que vocês descobriram:**

1. A taxa de no-show cresceu ao longo do tempo? Em que período a mudança foi mais acentuada?
2. O perfil dos pacientes (idade, tempo de serviço) mudou junto? Isso pode explicar parte da tendência?
3. Há períodos sem dados ou com pouquíssimos registros? Qual evento externo explica?
4. Que implicação essa deriva temporal tem para a validação do modelo preditivo do TCC?

---

1. Sim, e de forma acentuada: de **4,0% (2016) para 17,2% (2022)**. Os maiores saltos foram de 2016 para 2017
   (+4,3 p.p.) e de 2018 para 2019 (+4,3 p.p.). Depois disso a alta continua de forma gradual, inclusive durante a
   pandemia.

2. A idade mediana oscila pouco (6,4 a 8,1 anos), mas o **tempo de serviço mediano mais que triplica**, de 162 para
   597 dias. A população foi ficando mais "veterana", e os veteranos faltam mais (Atividade 5). Isso pode explicar
   parte da tendência. A **qualidade do registro também piora**: a especialidade ausente passa de 5% (2016) para 47%
   (2022), como mostra a fatia cinza no gráfico de composição.

3. Sim:
   - **7 meses sem nenhum registro** (2018-09, 2019-01, 2020-01, 2020-02, 2020-04, 2020-05, 2020-07);
   - 5 meses com menos de 100 agendamentos, concentrados na pandemia (2020), quando o atendimento presencial foi
     suspenso;
   - 2022 vai só até fevereiro.

   Por isso, as taxas de 2020 (904 registros) e 2022 (942) são instáveis. Os dias sem medição de clima também caem
   nesses períodos (nov–dez/2020 e jan–fev/2022). A temperatura deles foi preenchida com o Open-Meteo, mas a chuva
   continua ausente e, na prática, funciona como um indicador de "2022".

4. Há **deriva temporal (*concept drift*)**. Um k-fold aleatório mistura anos e mede um desempenho otimista. O TCC
   deve:
   - incluir o ano (ou período) como variável;
   - validar respeitando o tempo, por exemplo treinando em 2016–2019 e testando em 2021, além do agrupamento por
     paciente;
   - reportar as métricas por período.

   Um modelo calibrado no passado tende a **subestimar** o no-show atual.

---

## Atividade 7: Testes de Hipótese: Da Observação à Conclusão

### Briefing

Os gráficos sugeriram diferenças. São reais ou acaso?

- **H0 (hipótese nula):** não há diferença; **H1:** há diferença real; rejeita-se H0 se *p* < 0,05.
- **Variável numérica × dois grupos** (ex.: dias em serviço de quem falta × quem comparece): **Mann-Whitney U**,
  não paramétrico, porque as distribuições são assimétricas (Atividade 4). Para três ou mais grupos, usa-se
  **Kruskal-Wallis**.
- **Alvo binário × variável categórica** (ex.: taxa de no-show por clima): **qui-quadrado de independência**. O
  Mann-Whitney não serve aqui porque o "valor" de cada registro é só 0 ou 1.

**Atenção: significância ≠ relevância.** Com ~50 mil registros, diferenças ínfimas viram "significativas". Por isso,
cada teste vem acompanhado de um **tamanho de efeito**: correlação rank-bisserial para Mann-Whitney e **V de
Cramér** para qui-quadrado (≈0,1 pequeno, ≈0,3 médio, ≈0,5 grande).

> **Ponte com o TCC:** a H3 do projeto compara os modelos com o **teste de Wilcoxon pareado** entre folds da
> validação cruzada, que é o primo pareado do Mann-Whitney. O último bloco mostra uma limitação importante desse
> desenho.

In [ ]:
def mann_whitney(a, b):
    u, p = stats.mannwhitneyu(a, b, alternative="two-sided")
    return u, p, 2 * u / (len(a) * len(b)) - 1  # rank-bisserial: P(a > b) − P(a < b)

def cramers_v(x, y):
    tabela = pd.crosstab(x, y)
    chi2, p, gl, _ = stats.chi2_contingency(tabela)
    return chi2, p, gl, np.sqrt(chi2 / (tabela.to_numpy().sum() * (min(tabela.shape) - 1)))

# Teste 1 — H0: a distribuição de dias em serviço é igual entre quem falta e quem comparece
dados = analise[["dias_em_servico", "no_show"]].dropna().query("dias_em_servico >= 0")
falta, comparece = dados.query("no_show == 1").dias_em_servico, dados.query("no_show == 0").dias_em_servico
u, p, rb = mann_whitney(falta, comparece)
print("TESTE 1 — dias em serviço: no-show × compareceu")
print(f"  mediana no-show: {falta.median():.0f} dias | mediana compareceu: {comparece.median():.0f} dias")
print(f"  U = {u:,.0f} | p = {p:.2e} | rank-bisserial = {rb:+.3f}")
print("  Conclusão:", "rejeita H0" if p < 0.05 else "não rejeita H0", "| efeito", "desprezível" if abs(rb) < 0.1 else "pequeno" if abs(rb) < 0.3 else "médio/grande")

fig, ax = plt.subplots(figsize=(6, 3))
sns.boxplot(data=dados, x="dias_em_servico", y=dados.no_show.map({0: "compareceu", 1: "no-show"}), hue="no_show",
            palette=[BLUE, ORANGE], legend=False, ax=ax, fliersize=1, width=0.5, linewidth=1)
ax.set_ylabel(""); ax.set_title("Dias em serviço por desfecho")
plt.tight_layout(); plt.show()

In [ ]:
# Teste 2 — hipótese do grupo: o clima extremo (frio ou calor intenso) aumenta as faltas?
# H0: a taxa de no-show é a mesma em todas as categorias de heat_intensity
# H1: a taxa de no-show depende da intensidade térmica do dia
# Primeiro com a coluna ORIGINAL (clima ausente codificado como "heavy_cold"), depois com a coluna corrigida
# (32 dias preenchidos com a temperatura do Open-Meteo).
for rotulo, x, y in [("original", df.heat_intensity, df.no_show.eq("yes").astype(int)),
                     ("corrigida", analise.heat_intensity, analise.no_show)]:
    chi2, p, gl, v = cramers_v(x, y)
    tabela = y.groupby(x).agg(taxa="mean", n="size").sort_values("taxa")
    print(f"TESTE 2 — heat_intensity × no_show (coluna {rotulo})")
    print(tabela.assign(taxa=tabela.taxa.map("{:.1%}".format)).to_string())
    print(f"  χ² = {chi2:.1f} | gl = {gl} | p = {p:.2e} | V de Cramér = {v:.3f}\n")

# Controle: o efeito do clima sobrevive dentro de cada ano? (o frio pode estar concentrado em anos de taxa alta)
print("V de Cramér de heat_intensity (corrigida) × no_show dentro de cada ano:")
for ano_, g in analise.groupby("ano"):
    if len(g) > 2000:
        _, p_a, _, v_a = cramers_v(g.heat_intensity, g.no_show)
        print(f"  {ano_}: V = {v_a:.3f} (p = {p_a:.3f})")

In [ ]:
# Ranking de associação com o no-show
#  - numéricas: correlação de Spearman (equivale à ponto-bisserial com alvo 0/1)
#  - categóricas: V de Cramér (ausente = categoria própria)
ranking = []
for col in ["idade_consulta", "dias_em_servico", "hora", "average_temp_day", "max_temp_day", "average_rain_day", "max_rain_day"]:
    d = analise[[col, "no_show"]].dropna()
    rho, p = stats.spearmanr(d[col], d.no_show)
    ranking.append({"variável": col, "medida": "|Spearman|", "efeito": abs(rho), "p": p})
for col in ["ano", "appointment_time", "cid3", "city", "disability", "specialty", "dia_semana", "mes", "faixa_etaria",
            "heat_intensity", "rain_intensity", "appointment_shift", "patient_needs_companion", "gender"]:
    x = analise[col].astype("string")
    x = x.dropna() if col == "rain_intensity" else x.fillna("(ausente)")  # chuva ausente = dias sem medição (2022)
    _, p, _, v = cramers_v(x, analise.no_show[x.index])
    ranking.append({"variável": col, "medida": "V de Cramér", "efeito": v, "p": p})
ranking = pd.DataFrame(ranking).sort_values("efeito")

fig, ax = plt.subplots(figsize=(8, 6.5))
cores = [BLUE if m == "V de Cramér" else ORANGE for m in ranking.medida]
ax.barh(ranking.variável, ranking.efeito, color=cores, height=0.65)
for i, (e, p) in enumerate(zip(ranking.efeito, ranking.p)):
    ax.text(e, i, f" {e:.3f}{'' if p < 0.05 else ' (n.s.)'}", va="center", fontsize=7, color=INK)
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=BLUE), plt.Rectangle((0, 0), 1, 1, color=ORANGE)],
          labels=["categórica (V de Cramér)", "numérica (|Spearman|)"], frameon=False, fontsize=8, loc="lower right")
ax.set_title("Força da associação marginal com o no-show"); ax.grid(axis="y", visible=False)
ax.set_xlim(0, ranking.efeito.max() * 1.2)
plt.tight_layout(); plt.show()

> **Leitura cuidadosa do ranking:** o V de Cramér tende a ficar inflado em variáveis com muitas categorias
> (`appointment_time` tem 74 horários e `cid3` tem 46 códigos), porque categorias pequenas têm taxas extremas por
> acaso. E `ano` lidera por causa da deriva temporal, não por ser uma "causa" do no-show. Essas variáveis precisam
> ser agrupadas ou discretizadas antes de entrar na Rede Bayesiana.

In [ ]:
# Ponte com a H3 do TCC: qual é o menor p-valor possível no Wilcoxon pareado com k pares?
# Com todos os k pares favorecendo o mesmo cenário, p_min = 2 / 2^k (bilateral) ou 1 / 2^k (unilateral).
for k in [5, 6, 10, 20, 50]:
    melhor_caso = stats.wilcoxon(np.arange(1, k + 1), alternative="two-sided").pvalue
    print(f"k = {k:>2} pares → menor p bilateral = {melhor_caso:.3g}", "(nunca < 0,05!)" if melhor_caso >= 0.05 else "")

**Checkpoint: registre aqui o que vocês descobriram:**

1. A diferença de dias em serviço entre quem falta e quem comparece é estatisticamente significativa? E relevante na prática?
2. Qual hipótese o grupo testou no Teste 2? O resultado foi o esperado? O efeito sobreviveu ao controle por ano?
3. No ranking de associação, quais variáveis lideram? Isso faz sentido intuitivamente?
4. Por que usar Mann-Whitney (e não teste-t) para numéricas, e qui-quadrado para o alvo binário?
5. O que o último bloco revela sobre o desenho experimental da H3 (Wilcoxon com k_fold = 5)?

---

1. **Estatisticamente significativa, mas sem relevância prática.** O teste dá *p* = 7,5 × 10⁻¹¹, mas a correlação
   rank-bisserial é de apenas **+0,063** (efeito desprezível). As medianas são 294 dias (no-show) contra 273
   (compareceu), uma diferença de 3 semanas em um vínculo de quase um ano. Com 44 mil registros, até isso é
   "significativo".

2. A hipótese testada foi: **"o clima extremo aumenta as faltas"**. Com a coluna **original**, "frio intenso" parecia o
   pior cenário (14,3%), mas isso era o **artefato** dos dias sem medição. Com a coluna corrigida (e os 32 dias preenchidos pela temperatura real do Open-Meteo), o frio intenso cai para
   10,0%. O resultado real é diferente do esperado: **dias amenos têm menos faltas (8,5%) do que qualquer condição não
   amena (10–11,5%)**, e o extremo não é pior que o moderado. O "quente" sobe para 11,5% porque recebe os 850 registros
   de 2022, mais um sinal da deriva temporal. O efeito é muito pequeno (V = 0,042), mas
   **se mantém dentro da maioria dos anos** (V entre 0,036 e 0,062, significativo em 2017, 2018, 2019 e 2021; em
   2016 não é significativo). Fica uma
   ressalva: a estação do ano (mês) pode confundir esse resultado.

3. Lideram `ano` (V = 0,127), `appointment_time` (0,119), `cid3` (0,086), `city` (0,076), `mes`, `disability` e
   `faixa_etaria`. **Todas as associações são fracas** (V < 0,15; |ρ| ≤ 0,03 nas numéricas), e não existe um preditor
   isolado forte. O `ano` lidera por causa da deriva, e `appointment_time`/`cid3` têm o V inflado pela alta
   cardinalidade. O preditor mais promissor, o **histórico de faltas do paciente** (Atividade 5), nem aparece no
   ranking porque ainda precisa ser construído (notebook 02).

4. O **teste-t** compara médias e supõe normalidade, mas `dias_em_servico` e a idade são muito assimétricos, com
   outliers que arrastam a média. O **Mann-Whitney** compara postos (*ranks*) e não depende da forma da distribuição.
   Já com o **alvo binário**, cada registro vale só 0 ou 1, e comparar "postos" de 0/1 não faz sentido. A pergunta
   certa é se a proporção de faltas muda entre os grupos, e isso é exatamente o que o **qui-quadrado** testa na
   tabela de contingência.

5. **Com 5 folds, o Wilcoxon pareado nunca atinge *p* < 0,05.** O menor *p* bilateral possível é **0,0625**, mesmo que
   o framework vença nos 5 folds. Do jeito que está, a H3 **não pode ser confirmada**, independentemente dos
   resultados. Possíveis correções para a metodologia (Seção 3.10):
   - **validação cruzada repetida** (por exemplo, 10 × 5 = 50 pares, com *p* mínimo de ~10⁻¹⁵);
   - o **t-teste corrigido de Nadeau–Bengio**, próprio para folds correlacionados;
   - ou, no mínimo, o teste unilateral com 6 ou mais folds.

---

## Atividade 8: Relatório Executivo e Direções de Pesquisa

### Briefing

O relatório é para a **gestão do serviço de reabilitação**: pessoas que não sabem Python nem estatística. A
linguagem deve ser clara, com números concretos e sem jargão. Não basta dizer "o clima importa": é preciso dizer
**quanto**, **para quem** e **com que confiança**.

O bloco de **direções de pesquisa** conecta a EDA ao restante do TCC: o que os dados permitem modelar, o que falta
e como os próximos notebooks avançam.

In [ ]:
resumo = {
    "agendamentos": len(analise),
    "taxa_no_show": analise.no_show.mean(),
    "pacientes_aprox": analise.patient_id.nunique(),
    "no_show_2016": anual.loc[2016, "taxa_no_show"],
    "no_show_2022": anual.loc[2022, "taxa_no_show"],
    "top10_pacientes_%faltas": acum[int(0.10 * len(acum)) - 1],
    "pacientes_nunca_faltaram": (por_paciente.faltas == 0).mean(),
}
pd.Series(resumo).to_frame("valor")

### Relatório Executivo

*Para a gestão do serviço de reabilitação: linguagem direta, números concretos.*

---

**1. Qualidade dos dados**

A base tem quase 50 mil agendamentos em seis anos, mas com lacunas importantes:

- **1 em cada 10 agendamentos não tem nenhum dado do paciente**, e justamente esses faltam mais (15% contra 9%).
- A coluna de idade foi calculada na data em que a base foi extraída, não na data da consulta: erra em até 6 anos
  nas consultas antigas e foi recalculada.
- Nos dias sem medição de clima (quase todos em pleno verão de 2022), o sistema registrou "frio intenso", o que
  distorceria qualquer conclusão sobre o clima. A temperatura real desses dias foi recuperada de uma base
  meteorológica pública (Open-Meteo), conferida contra os dias medidos.
- Não há um código de paciente nem a data em que a consulta foi marcada. Por isso, não dá para medir o tempo de espera.
- O campo de motivo da falta fica vazio em 2 de cada 3 faltas.
- O que hoje se registra como "falta" às vezes inclui cancelamentos avisados.

**Limitação:** as conclusões valem para os agendamentos com cadastro; o grupo sem cadastro é exatamente o de maior risco.

---

**2. O tamanho do problema**

**Cerca de 1 em cada 10 consultas (9,7%) termina em falta**, o que equivale a mais de 4.800 horários perdidos no
período. E o problema está **crescendo**: a taxa de faltas **quadruplicou**, de 4% em 2016 para 17% no início de 2022.

---

**3. Quem falta mais**

As faltas **não se espalham igualmente**: **10% dos pacientes concentram 42% de todas as faltas**, enquanto mais de
um terço dos pacientes nunca faltou. Faltam mais:

- pacientes sem cadastro completo (15%);
- moradores de municípios mais distantes, como Luiz Alves e Ilhota (13–14%), contra 6% em Penha;
- pacientes com mais de 14 meses de tratamento (12%);
- crianças de 6 a 11 anos que dependem de acompanhante (10,5% contra 7,9% das que não dependem).

---

**4. Quando se falta mais**

- **No fim da tarde:** às 17h a taxa é de 12%, e o horário das 17:40 chega a **21%**, o dobro da média. Às 18h cai
  para 7%.
- **Às sextas e segundas**, em relação às terças.
- **Em junho e fevereiro** (12–13%) contra julho e agosto (7%).

---

**5. Clima**

O clima tem **efeito pequeno**: dias de temperatura amena têm cerca de 2 pontos percentuais a menos de faltas do que
dias frios ou quentes. A chuva praticamente não muda o comparecimento. **O clima não deve ser a prioridade das ações.**

---

**6. Recomendações ao serviço**

1. **Foco nos pacientes reincidentes:** lembretes ativos e contato prévio para os ~10% que mais faltam já atacam
   quase metade do problema.
2. **Revisar a grade de fim de tarde** (17h–17h40), por exemplo com confirmação no dia ou *overbooking* controlado
   nesses horários.
3. **Completar o cadastro** dos pacientes sem dados e **registrar sempre o motivo** da falta, separando "faltou sem
   avisar" de "cancelou com antecedência".
4. **Acompanhar a taxa mês a mês**: a tendência de alta é o sinal mais forte da base.

---

**7. Direções de pesquisa (próximas etapas do TCC)**

- **Histórico de faltas** do paciente, calculado só com consultas anteriores: deve ser o preditor mais forte e ainda
  não foi construído.
- **Ajuste da metodologia:**
  - validação agrupada por paciente e ciente do tempo;
  - validação cruzada repetida, porque o Wilcoxon com 5 folds não consegue atingir *p* < 0,05;
  - substituir o "tempo de espera fracionado" (dado inexistente) por tempo de vínculo e histórico.
- **Rede Bayesiana e Manto de Markov** para separar efeitos que hoje se confundem: tempo de vínculo × ano,
  acompanhante × idade, clima × estação.
- **Com mais dados:** a data de marcação (para medir a antecedência), a distância até o serviço e o histórico de
  lembretes enviados enriqueceriam o modelo.

---

**Do diagnóstico ao modelo (próximos notebooks do TCC):**

| Notebook | O que responde | Achado da EDA que o motiva |
|----------|----------------|----------------------------|
| `01_diagnostico` | Qualidade, vazamento e estrutura de validação | Atividades 2, 3 e 6 |
| `02_engenharia_variaveis` | Histórico de faltas, tempo de vínculo, idade corrigida, discretização | Atividades 3, 4 e 5 |
| `03_rede_bayesiana` | Quais variáveis formam o Manto de Markov do no-show? | Atividade 7 (associação marginal × condicional) |
| `04_experimentos` | O framework de normalização + SMOTENC melhora ROC-AUC/PR-AUC? | Atividades 4, 6 e 7 (escalonador, validação, Wilcoxon) |